# SAE Sweep Analysis

Compares SAE runs across the hyperparameter sweep to choose the best `k` and `n_latent` per layer.

**Prerequisites:** Run `scripts/eval_sweep.py` once to generate `sweep_metrics.csv`:
```bash
python scripts/eval_sweep.py --sweep_dir /path/to/sweep --data_dir /path/to/data
```

**Key SAE metrics:**

| Metric | What it tells you | Goal |
|---|---|---|
| **Variance explained** | How much of the original signal the SAE reconstructs | Higher is better (>=0.90 is good) |
| **MSE** | Raw reconstruction error per dimension | Lower is better |
| **Dead features (%)** | Fraction of latent features that never activate | Lower is better (<10% ideal) |

**Trade-offs:**
- **k** (sparsity): Lower k = more interpretable but worse reconstruction
- **n_latent** (dictionary size): Larger = more capacity but more dead features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Update this path ---
CSV_PATH = "/ocean/projects/atm170004p/jshen6/cae_sae/sweep_metrics.csv"

df = pd.read_csv(CSV_PATH)

LAYERS = ["IN", "E1", "E2", "E3", "E4", "E5", "D1", "D2", "D3", "D4", "D5", "OUT"]
K_VALUES = sorted(df["k"].unique())
N_LATENT_VALUES = sorted(df["n_latent"].unique())

print(f"Loaded {len(df)} runs: {len(df['layer'].unique())} layers, k={K_VALUES}, n_latent={N_LATENT_VALUES}")
df.head()

## 1. Variance Explained vs. k

The primary quality metric. A good SAE should achieve >=0.90 (90%).

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12), sharey=True)

for i, layer in enumerate(LAYERS):
    ax = axes.flat[i]
    sub = df[df["layer"] == layer]
    if sub.empty:
        ax.set_title(f"{layer} (no data)")
        continue

    for n_latent in N_LATENT_VALUES:
        s = sub[sub["n_latent"] == n_latent].sort_values("k")
        ax.plot(s["k"], s["variance_explained"], "o-", label=f"n={n_latent}")

    ax.set_title(layer, fontweight="bold")
    ax.set_xlabel("k")
    ax.set_xticks(K_VALUES)
    ax.axhline(0.90, color="gray", ls="--", alpha=0.5)
    if i == 0:
        ax.legend(fontsize=8)
    if i % 4 == 0:
        ax.set_ylabel("Variance Explained")

fig.suptitle("Variance Explained vs. k", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 2. Dead Features vs. k

Dead features are wasted capacity. Green line = 10% (good), red line = 30% (concerning).

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12), sharey=True)

for i, layer in enumerate(LAYERS):
    ax = axes.flat[i]
    sub = df[df["layer"] == layer]
    if sub.empty:
        ax.set_title(f"{layer} (no data)")
        continue

    for n_latent in N_LATENT_VALUES:
        s = sub[sub["n_latent"] == n_latent].sort_values("k")
        ax.plot(s["k"], s["dead_pct"], "o-", label=f"n={n_latent}")

    ax.set_title(layer, fontweight="bold")
    ax.set_xlabel("k")
    ax.set_xticks(K_VALUES)
    ax.axhline(10, color="green", ls="--", alpha=0.5)
    ax.axhline(30, color="red", ls="--", alpha=0.5)
    if i == 0:
        ax.legend(fontsize=8)
    if i % 4 == 0:
        ax.set_ylabel("Dead Features (%)")

fig.suptitle("Dead Feature % vs. k", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 3. Reconstruction vs. Sparsity (annotated with dead%)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))

for i, layer in enumerate(LAYERS):
    ax = axes.flat[i]
    sub = df[df["layer"] == layer]
    if sub.empty:
        ax.set_title(f"{layer} (no data)")
        continue

    for n_latent in N_LATENT_VALUES:
        s = sub[sub["n_latent"] == n_latent].sort_values("k")
        ax.plot(s["k"], s["variance_explained"], "o-", label=f"n={n_latent}")
        for _, row in s.iterrows():
            ax.annotate(f"{row['dead_pct']:.0f}%",
                       (row["k"], row["variance_explained"]),
                       textcoords="offset points", xytext=(5, 5), fontsize=6, alpha=0.7)

    ax.set_title(layer, fontweight="bold")
    ax.set_xlabel("k")
    ax.set_ylabel("Variance Explained")
    ax.set_xticks(K_VALUES)
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle("Reconstruction vs. Sparsity (annotations = dead%)", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 4. Summary Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for idx, n_latent in enumerate(N_LATENT_VALUES):
    ax = axes[idx]
    sub = df[df["n_latent"] == n_latent]
    if sub.empty:
        continue

    pivot = sub.pivot(index="layer", columns="k", values="variance_explained")
    pivot = pivot.reindex([l for l in LAYERS if l in pivot.index])

    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=1.0)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("k")
    ax.set_title(f"n_latent = {n_latent}")

    for ii in range(len(pivot.index)):
        for jj in range(len(pivot.columns)):
            val = pivot.values[ii, jj]
            if not np.isnan(val):
                ax.text(jj, ii, f"{val:.3f}", ha="center", va="center", fontsize=8,
                       color="black" if val > 0.7 else "white")

fig.colorbar(im, ax=axes, label="Variance Explained", shrink=0.8)
fig.suptitle("Variance Explained — All Layers x k x n_latent", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for idx, n_latent in enumerate(N_LATENT_VALUES):
    ax = axes[idx]
    sub = df[df["n_latent"] == n_latent]
    if sub.empty:
        continue

    pivot = sub.pivot(index="layer", columns="k", values="dead_pct")
    pivot = pivot.reindex([l for l in LAYERS if l in pivot.index])

    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=50)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("k")
    ax.set_title(f"n_latent = {n_latent}")

    for ii in range(len(pivot.index)):
        for jj in range(len(pivot.columns)):
            val = pivot.values[ii, jj]
            if not np.isnan(val):
                ax.text(jj, ii, f"{val:.0f}%", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=axes, label="Dead Features (%)", shrink=0.8)
fig.suptitle("Dead Features % — All Layers x k x n_latent", fontsize=14)
fig.tight_layout()
plt.show()

## 5. Recommended Configurations

Picks the sparsest config per layer that achieves >= 90% variance explained.

In [ ]:
VE_THRESHOLD = 0.90

recommendations = []
for layer in LAYERS:
    sub = df[(df["layer"] == layer) & (df["variance_explained"] >= VE_THRESHOLD)]
    if sub.empty:
        sub = df[df["layer"] == layer]
        if sub.empty:
            continue
        best = sub.loc[sub["variance_explained"].idxmax()]
        recommendations.append({
            "layer": layer, "k": int(best["k"]), "n_latent": int(best["n_latent"]),
            "VE": best["variance_explained"], "dead%": best["dead_pct"],
            "note": "below threshold",
        })
    else:
        best = sub.sort_values(["k", "dead_pct"]).iloc[0]
        recommendations.append({
            "layer": layer, "k": int(best["k"]), "n_latent": int(best["n_latent"]),
            "VE": best["variance_explained"], "dead%": best["dead_pct"], "note": "",
        })

rec_df = pd.DataFrame(recommendations)
print(f"Recommended configs (VE >= {VE_THRESHOLD}, prefer lowest k then lowest dead%):\n")
print(rec_df.to_string(index=False))

## Interpretation Guide

1. **Variance explained heatmap**: If all configs for a layer are below 0.90, the SAE may need more epochs or a different architecture.

2. **Diminishing returns in k**: If k=32 to k=64 barely improves VE, use k=32 (more interpretable).

3. **Dictionary size**: If n=4096 and n=8192 give similar VE but 8192 has more dead features, the extra capacity is wasted.

4. **Layer-specific behavior is expected**: Early/late layers (IN, OUT) often differ from bottleneck layers (E5, D1).

5. **Recommendations are a starting point**: Push k lower for interpretability, allow higher k for faithful reconstruction.